# TD 3 — Corrigé

**Analyse des données — L3 Économie**

Mis en ligne après la séance.

## Partie 1 — Charger, et ouvrir le journal

In [ ]:
import pandas as pd
import numpy as np

URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/cm03-salaires.csv")

brut = pd.read_csv(URL, sep=";", encoding="utf-8")
df   = brut.copy()

def en_nombre(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(" ", "", regex=False)
                         .str.replace(",", ".", regex=False),
        errors="coerce")

df["SALAIRE_NET"] = en_nombre(df["SALAIRE_NET"])
df["PRIME"]       = en_nombre(df["PRIME"])

print(df.shape)
print("Salaire moyen de depart :", round(df["SALAIRE_NET"].mean(), 2))

In [ ]:
journal = []

def noter(etape, decision, justification, data):
    journal.append({
        "etape": etape, "decision": decision,
        "lignes": len(data),
        "salaire_moyen": round(data["SALAIRE_NET"].mean(), 2),
        "justification": justification})
    return pd.DataFrame(journal)

noter("Chargement", "—", "Fichier brut", df)

**Réponses 1 et 2.** 8 012 lignes, salaire moyen **67 745 €**.

Le salaire net mensuel moyen publié par l'INSEE est de l'ordre de 2 600 €. Le chiffre du fichier est donc **26 fois plus élevé** — un ordre de grandeur impossible, repérable en dix secondes sans ordinateur.

> C'est la première compétence de la séance : confronter un résultat à ce qu'on sait du monde. Aucune ligne de code ne l'aurait détecté.

**Sur le journal.** Il se tient **pendant**, pas après.

Rempli de mémoire à la fin, il donne des chiffres incohérents entre étapes : on ne se souvient plus de l'ordre, ni de ce que valait la moyenne avant telle correction. Or c'est précisément l'effet de chaque décision qu'on cherche à mesurer.

Ouvrir le journal en premier coûte trois lignes. Le reconstituer après coûte une heure, et il est faux.

## Partie 2 — Les manquants qui n'en ont pas l'air

In [ ]:
print(df.isna().sum())
print()
print(df["SALAIRE_NET"].value_counts().head(5))

**Réponses 3 à 5.** `isna()` ne signale rien — et pourtant **525 observations** portent la valeur `999 999,00`.

Ce n'est pas un salaire : c'est un **code de non-réponse**. Le dictionnaire l'indique. Pandas, lui, le traite comme un nombre et l'intègre à la moyenne.

In [ ]:
def retirer_non_reponse(serie, code):
    """Met en manquant le code annonce par le dictionnaire."""
    return serie.replace(code, np.nan)

for colonne in ["SALAIRE_NET", "PRIME"]:
    df[colonne] = retirer_non_reponse(df[colonne], 999999.0)

noter("Code 999999", "Conversion", "Dictionnaire : non-reponse", df)

**Réponses 6 et 7.** Le salaire moyen passe de **67 745 €** à **2 374 €**.

Un écart de 65 371 euros, soit un facteur **28**.

Et surtout : **aucun message d'erreur, ni avant ni après**. Le calcul initial était parfaitement valide du point de vue de Python, et absurde du point de vue économique.

> C'est le type d'erreur le plus dangereux : celui qui produit un nombre plausible pour une machine et invraisemblable pour un économiste. Seul un ordre de grandeur en tête permet de le repérer.

In [ ]:
df["MOTIF_ABS"].value_counts(dropna=False).sort_index()

**Réponses 8 et 9.** Sur `MOTIF_ABS`, la non-réponse est codée **99** — 32 observations. Le code **9** est une **modalité réelle**, documentée, qui concerne **61 observations**.

Un `replace(9, np.nan)` appliqué par habitude détruirait donc ces 61 situations, silencieusement, en les transformant en non-réponses.

**La règle :** le code de non-réponse est un *argument* de la fonction, jamais une constante cachée dedans. C'est ce qui permet de traiter chaque variable selon ce que son dictionnaire annonce.

## Partie 3 — Zéro n'est pas manquant

In [ ]:
print((df["SALAIRE_NET"] == 0).sum(), "salaires nuls")
print()
print(pd.crosstab(df.loc[df["SALAIRE_NET"] == 0, "STATUT"],
                  df.loc[df["SALAIRE_NET"] == 0, "CONTRAT"]))

**Réponse 10.** **675 salaires nuls**, et le croisement ne laisse aucune ambiguïté : *tous* ont `STATUT = 3` et `CONTRAT = 8`.

Une seule case du tableau est remplie. Ce ne sont donc ni des non-réponses dispersées, ni des erreurs de saisie : c'est une **catégorie homogène**, identifiée par le dictionnaire.

Le contraste avec le code 999 999 mérite d'être souligné : là, une valeur aberrante ; ici, un zéro parfaitement légitime qui décrit une situation réelle.

In [ ]:
moyenne_avec = df["SALAIRE_NET"].mean()
moyenne_sans = df.loc[df["SALAIRE_NET"] > 0, "SALAIRE_NET"].mean()

print("Avec les zeros :", round(moyenne_avec, 2))
print("Sans les zeros :", round(moyenne_sans, 2))
print("Ecart          :", round(moyenne_sans - moyenne_avec, 2), "euros")

**Réponse 11.** **2 374 €** avec, **2 610 €** sans. Un écart de 236 euros, soit 10 %.

Les deux chiffres sont exacts ; ils répondent à deux questions différentes.

- **Avec les zéros** : quel est le revenu d'activité moyen de l'ensemble des personnes du fichier ?
- **Sans les zéros** : quel est le salaire moyen de celles qui en perçoivent un ?

La seconde correspond à ce qu'on appelle ordinairement « le salaire moyen ». Mais la décision doit être **écrite**, parce qu'un lecteur qui l'ignore ne peut pas comparer votre chiffre à un autre.

> Le champ d'un indicateur n'est pas une formalité : c'est la moitié de sa définition.

In [ ]:
df = df.loc[df["SALAIRE_NET"].isna() | (df["SALAIRE_NET"] > 0)].copy()
noter("Salaires nuls", "Exclusion", "Hors champ : STATUT 3, CONTRAT 8", df)

## Partie 4 — La censure haute

In [ ]:
print(df["SALAIRE_NET"].quantile([0.90, 0.95, 0.99, 0.999]).round(2))
print()
print((df["SALAIRE_NET"] == 10000).sum(), "observations a 10 000 euros")

**Réponse 12.** Les quantiles à 99 % et 99,9 % sont **égaux** — tous deux à 10 000 euros. Sur une variable continue, c'est impossible autrement que par un plafond.

**72 observations** portent exactement cette valeur. Le dictionnaire le confirme : les salaires supérieurs sont ramenés à 10 000 euros, pour préserver l'anonymat.

La distribution est **censurée à droite**. On ne connaît pas les hauts salaires : on sait seulement qu'ils dépassent le plafond.

**Réponse 13.** Une formulation acceptable :

> *Les salaires nets mensuels supérieurs à 10 000 euros sont ramenés à ce plafond, ce qui concerne 72 observations, soit environ 1 % de l'échantillon. Les indicateurs de dispersion et d'inégalité présentés constituent donc des bornes inférieures.*

Trois éléments obligatoires : le **plafond**, la **part concernée**, et la **nature de la borne**.

Ce que la censure interdit d'affirmer : tout ce qui porte sur le haut de la distribution. Un écart-type, un rapport interdécile, un indice de Gini sont tous sous-estimés — et le sens de l'erreur est connu, même si son ampleur ne l'est pas.

## Partie 5 — Le journal

In [ ]:
pd.DataFrame(journal)

**Réponse 14.** De **67 745 €** à **2 610 €** : la moyenne a été divisée par 26.

L'étape « salaires nuls » est celle qui mérite le commentaire. Elle ne change la moyenne que de 10 % — faible au regard du facteur 28 de l'étape précédente — mais elle change **la définition de l'indicateur**. Avant, on mesurait un revenu d'activité moyen sur l'ensemble du fichier ; après, un salaire moyen sur les seuls salariés.

> Une étape peut être indispensable sans être spectaculaire. L'effet sur le chiffre et l'effet sur le sens sont deux choses distinctes.

## Pour ceux qui ont terminé

In [ ]:
print(brut.duplicated().sum(), "lignes identiques partout")
print(brut.duplicated(subset=["IDENT"]).sum(), "IDENT repetes")
print(brut.duplicated(subset=["IDENT", "NOPOSTE"]).sum(), "couples IDENT + NOPOSTE repetes")

**Réponse 15.** Trois comptages, trois choses différentes.

- **12 lignes identiques partout** : de vrais doublons techniques, à supprimer.
- **632 `IDENT` répétés** : ce ne sont **pas** des doublons. `IDENT` identifie une personne, et `NOPOSTE` son poste — une même personne peut occuper plusieurs postes dans l'année.
- **12 couples répétés** : ce sont exactement les lignes identiques.

**La clé du fichier est le couple `IDENT` + `NOPOSTE`.**

Supprimer les « doublons » d'`IDENT` aurait retiré 632 postes réels. L'erreur ne produit aucun message : juste un échantillon qui n'est plus celui qu'on croit.

Et une nuance qui compte : si la question portait sur les **personnes** plutôt que sur les postes, il ne faudrait pas supprimer ces lignes mais les **agréger** — sommer les salaires par `IDENT`. Ce n'est pas la même opération.

In [ ]:
impute = df["SALAIRE_NET"].fillna(df["SALAIRE_NET"].mean())

print("moyenne  sans imputation :", round(df["SALAIRE_NET"].mean(), 2),
      "| avec :", round(impute.mean(), 2))
print("ecart-type sans          :", round(df["SALAIRE_NET"].std(), 2),
      "| avec :", round(impute.std(), 2))

**Réponse 16.** La moyenne est **inchangée** — 2 609,57 dans les deux cas. C'est mécanique : on impute précisément la moyenne.

L'écart-type, lui, **baisse** : de 1 464,77 à 1 411,38, soit 3,6 %. On a ajouté 525 observations sans aucune variabilité, et la dispersion s'en trouve diluée.

La baisse paraît modeste parce que les valeurs imputées ne représentent que 7 % de l'échantillon. **Le mécanisme, lui, est proportionnel** : avec un tiers de valeurs manquantes, l'écart-type serait amputé d'un tiers.

Les erreurs-types calculées ensuite seraient trop petites, et l'inférence trop confiante — le calcul croit disposer de plus d'information qu'il n'en existe.

> Imputer par la moyenne préserve ce qu'on regarde et abîme ce qu'on ne regarde pas. C'est ce qui rend cette pratique si tenace.

**Réponse 17.** Le test de robustesse ne cherche pas la bonne réponse : il vérifie que la conclusion ne dépend pas d'un arbitrage discutable.

La décision la plus contestable est ici l'exclusion des salaires nuls. Selon ce que l'on veut mesurer, elle se défend ou non — et la conclusion sur le niveau moyen y est sensible à hauteur de 10 %.

## Erreurs fréquentes

| Erreur | Symptôme | Ce qu'elle enseigne |
|---|---|---|
| Ne pas convertir les montants en nombres | `TypeError` sur `.mean()` | Une colonne de texte n'est pas une colonne de nombres |
| Garder le code 999 999 | salaire moyen à 67 745 € | Un nombre plausible pour la machine peut être absurde |
| `replace(9)` généralisé | 61 observations détruites | Le code de non-réponse est un argument, pas une constante |
| Traiter les zéros comme manquants | on perd une catégorie homogène | Zéro n'est pas manquant |
| Supprimer les `IDENT` répétés | 632 postes disparaissent | Un identifiant répété n'est pas un doublon |
| Imputer par la moyenne | écart-type effondré | Imputer est un modèle, pas une réparation |
| Journal rempli à la fin | chiffres incohérents | Le journal se tient pendant |